# N142 · 折半搜索与指数优化

**目标：** 把全集枚举拆成两侧并正确合并。

**先修：** N125, N042, N101。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 半集子集和；排序/二分；按元素个数分桶；输出规模；时间空间代价。

**配套讲解来源：** [同名逐章意见](../../comment/142_meet_in_the_middle.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

子集型问题（"从数组里挑一些数，让它们的和尽量接近目标"）最粗暴的解法是把 2^n 个子集全列一遍。n 小的时候没事，但 n 到了 40 左右，2^40 ≈ 一万亿，超算也要跑很久；而 2^20 ≈ 一百万，家用电脑毫秒级。折半搜索（meet in the middle）就是利用这个落差：把数组切成左右两半，各自分别枚举（每半只有 2^(n/2) 个方案），再用"排序 + 二分"把两半的方案配对起来，代替把全部方案两两配对。

具体到数字的例子（“自动测试”部分 的第一条测试）：输入 `nums=[5,-7,3,5]`，`goal=6`，问"某个子集的和与 6 的差最少是多少"。答案是 0，因为子集 {5, -7, 3, 5} 的和恰好等于 6。完整枚举要做 2^4=16 个子集，看着不多；但同一套框架放到 n=40 上，完整枚举是 2^40 次方级别，折半后只有约 2·2^20 加上排序二分，天壤之别。

本章实现两个目标不同的版本：

1. `min_abs_difference(nums, goal)`：任意大小的子集，和与 goal 的最小绝对差。
2. `minimum_difference_equal_size(nums)`：把数组分成**元素个数相同**的两半，让两组各自的总和之差最小（对应 LeetCode 2035）。拿 [3,9,7,3] 举例，合法分法（每边 2 个）一共有 3 种本质不同的拆法：

   | A 组 | B 组 | 和 A | 和 B | 差 |
   |------|------|------|------|----|
   | {3,9} | {7,3} | 12 | 10 | 2 |
   | {3,7} | {9,3} | 10 | 12 | 2 |
   | {3,3} | {9,7} | 6 | 16 | 10 |

   最小差是 2，这就是 “运行示例”部分 要复现的答案。

## 2. 基础知识：先读懂这些词

- **子集和（subset sum）**：从数组里挑出一个子集，把挑中的数加起来得到的值。不同子集可能算出相同的和（比如 [3,3] 里挑任何一个 3 都是 3），这很重要，后面要提。
- **折半搜索（meet in the middle）**：从正中间切开数组，左右各自枚举所有子集，然后把"右半的哪些子集和"排序存起来，左半每个方案去右半里二分找最佳搭档。前提是"目标"能拆成左右两半各贡献一部分、可加可减的量。
- **排序 + 二分（bisect）**：`bisect_left(arr, x)` 返回 arr 中第一个 ≥ x 的位置。排序一次 O(m log m)，之后每次查询 O(log m)，比每次线性扫快。
- **前驱/后继夹逼**：在有序数组里离目标最近的值，只可能是"第一个 ≥ 目标的"（后继）或"它前面那个"（前驱）两者之一，别的只会更远。所以二分后只检查 p-1 和 p 两个位置就够了。
- **按元素个数分桶（bucketing by count）**：在"两组必须一样大"的问题里，光知道右半的和还不够，还得知道它是挑了几个元素凑出来的；只有"左边挑 count 个 + 右边挑 half−count 个"的组合才合法。所以右半的子集和按"选中个数"分桶存放。
- **整数二倍表达式**：比较"两个和谁更接近总量一半"时，写成 `total - 2*(a+b)` 而不是 `(total)/2 - (a+b)`。前者全程整数运算，10^30 这种大数也不会丢精度；后者一旦除法介入变成浮点，大数就废了。
- **输出规模/仍是指数算法**：折半只是把指数从 n 砍到 n/2，复杂度还是 O(n·2^(n/2))，并没有变成多项式——它救的是 n≈40 这一档。

## 3. 思路推导：从小例子开始

**Step 1：会算"半个数组的全部子集和"吗？会。** `subset_sums` 用滚动扩充：手算 `subset_sums([3,9])`——起始 `[0]`（空集）；处理 3，把现有每个和 +3 追加进去，得 `[0,3]`；处理 9，得 `[0,3,9,12]`。这 4 个正是 [3,9] 的全部子集和（空集 0、{3}、{9}、{3,9}）。

**Step 2：两半配对（无个数限制版）。** 完整子集 = 左半子集 + 右半子集，所以"找和最接近 goal 的子集"等价于：左半每个和 a，去右半找和 b 使 a+b 最接近 goal，也就是 b 最接近 goal−a。右半排序后二分。

手算一遍 “自动测试”部分 的 `[5,-7,3,5], goal=6`：mid=2，左半 [5,−7] 的子集和是 `[0,5,-7,-2]`，右半 [3,5] 的子集和排序后是 `[0,3,5,8]`。

- a=0：找最接近 6 的 b。bisect 定位到 5 和（不存在的）8 之间，检查前驱 5 → |0+5−6|=1，answer 从 6 降到 1。
- a=5：找最接近 1 的 b。前驱 0 → |5+0−6|=1；后继 3 → |5+3−6|=2。
- a=−7：找最接近 13 的 b，最大只有 8 → |−7+8−6|=5。
- a=−2：找最接近 8 的 b，后继正好 8 → |−2+8−6|=0，answer 归零，立即返回 0。

这个 0 对应的完整子集是 {5,−7} ∪ {3,5}，和恰好 6。

**Step 3：等大小版为什么要分桶？** “运行示例”部分 的例子 `a=[3,9,7,3]`（总量 total=22，half=2）。如果只按和排序配对，可能配出"左边挑 2 个、右边挑 0 个"这种人数不合法的方案。所以右半按"选中个数"分桶：手算 `_sums_by_size([3,9])` 得到桶 `[[0], [3,9], [12]]`（挑 0 个 → 和 0；挑 1 个 → 3 或 9；挑 2 个 → 12）；`_sums_by_size([7,3])` 得到 `[[0], [7,3], [10]]`。这正好就是 “运行示例”部分 表格展示的四行数据（左 0 个配右 2 个、左 1 个配右 1 个……）。

**Step 4：桶间配对 + 二倍技巧。** 左边挑了 count 个（和为 a），就只能去右半第 half−count 号桶里找 b。要让两组总和之差小，就是让 a+b 尽量接近 total/2，即最小化 `|total − 2(a+b)|`。代码里目标值写成 `(total−2*a+1)//2`，全程整数。手算 a=3（count=1）：目标 = (22−6+1)//2 = 8，右半 1 号桶排序为 [3,7]，后继 7 → |22−2·(3+7)|=2。把所有桶轮一遍：count=0 时 a=0 配 b=10 得 |22−20|=2；count=2 时 a=12 配 b=0 得 |22−24|=2。最终答案 2，和 “运行示例”部分 打印的 `等大小最小差 2` 一致。人工验算：分法 {3,7} 和 {9,3}，两组和 10 与 12，差正好 2。

**Step 5：把 `_sums_by_size` 的循环也手算一遍**（它是最容易看错的一个函数）。输入 [7,3]：初始 buckets=[[0],[],[],[]]。处理 i=0（x=7）：count 从 1 倒着到 1，buckets[1] 从 buckets[0] 扩出 [0+7]=[7]，得 [[0],[7],[],[]]。处理 i=1（x=3）：count=2 先做，buckets[2] 从 buckets[1] 扩出 [7+3]=[10]；count=1 后做，buckets[1] 变 [7, 0+3]=[7,3]，得 [[0],[7,3],[10]]——正是 Step 3 的右半结果。注意 count=1 这步读的 buckets[0] 还是上一轮的值，而 buckets[1] 虽在本轮被改过，却是被"更高号"先改、再轮到自己当源时用的是旧的自己——倒序循环保证了"每个数至多带一次"的 0/1 语义。

**Step 6：分治感——为什么先左后右都行？** 两个函数的枚举顺序（左半逐个 a、右半整体排序）无所谓，因为配对判据只依赖 a 和目标值，左右之间没有顺序耦合。真正不可交换的是"排序必须发生在二分之前"——`min_abs_difference` 里若忘了 `sorted(...)`，bisect 在无序列表上的返回值毫无意义，这正是新手最常翻车的一行。

## 4. 核心代码：subset_sums

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def subset_sums(nums):
    sums = [0]
    for x in nums:
        sums += [s + x for s in sums]
    return sums
```

### 逐段读懂代码

### 函数一：`subset_sums(nums)` —— 半集子集和

```python
def subset_sums(nums):
    sums = [0]
    for x in nums:
        sums += [s + x for s in sums]
    return sums
```

从只含空集和 0 的列表起步；每处理一个新数 x，就"把现有每个和都加 x 得到一批新和"追加进列表——新一批是"包含 x 的子集"，旧一批是"不包含 x 的子集"，合起来恰是当前所有子集。注意它保留重复：[3,3] 会产出 [0,3,3,6] 而不是去重的 [0,3,6]。“正确性与复杂度”部分 特意提醒：它是"子集和的列表"，不是"不同和的集合"，别拿它当后者用（好在本章只对它做 min/二分，重复不碍事）。

### 函数二：`min_abs_difference(nums, goal)` —— 任意子集版最近和

```python
def min_abs_difference(nums, goal):
    mid = len(nums) // 2
    left = subset_sums(nums[:mid])
    right = sorted(subset_sums(nums[mid:]))
    answer = abs(goal)
```

切半、枚举左半（保持无序即可，反正要逐个遍历）、右半排序（为二分做准备）。answer 先用 |goal| 垫底——它恰好等于"左半空集 a=0 配右半空集 b=0"的差，是一个合法但通常不优的起点。

```python
for a in left:
        p=bisect_left(right,goal-a)
        for j in (p-1,p):
            if 0<=j<len(right): answer=min(answer,abs(a+right[j]-goal))
        if answer==0: return 0
    return answer
```

对每个左和 a：右半要找的是最接近 goal−a 的 b。`bisect_left` 给出第一个 ≥ 目标的位置 p，前驱 p−1 是最后一个 < 目标的；`0<=j<len(right)` 挡住 p 落在数组两端时的越界下标。夹逼这两个位置必含最近值。一旦差到 0 就不可能更小了，提前返回是纯粹的剪枝。

### 函数三：`_sums_by_size(nums)` —— 按"挑了几个"分桶的子集和

```python
def _sums_by_size(nums):
    buckets = [[] for _ in range(len(nums) + 1)]
    buckets[0] = [0]
    for i, x in enumerate(nums):
        for count in range(i + 1, 0, -1):
            buckets[count].extend((s + x for s in buckets[count - 1]))
    return buckets
```

buckets[k] 装的是"恰好挑 k 个元素"的全部和。处理第 i 个数（下标 i，即前 i+1 个数）时，"挑 count 个"的新方案只能来自"用前 i 个数挑了 count−1 个"再带上 x。内层循环从 i+1 **倒着**扫到 1，是为了不覆盖本轮还要读的低号桶（先更新高号桶，读的还是上一轮的低号桶内容）——这是 0/1 背包一维优化的同款技巧。

### 函数四：`minimum_difference_equal_size(nums)` —— 等大小划分最小差

```python
def minimum_difference_equal_size(nums):
    if len(nums) % 2:
        raise ValueError('even input length required')
    half = len(nums) // 2
    left = _sums_by_size(nums[:half])
    right = _sums_by_size(nums[half:])
    total = sum(nums)
    answer = float('inf')
```

奇数长度没法平分，直接报错。左右各分桶，total 记总和。答案初值正无穷。

```python
for count,values in enumerate(left):
        candidates=sorted(right[half-count])
        for a in values:
            target=(total-2*a+1)//2; p=bisect_left(candidates,target)
            for j in (p-1,p):
                if 0<=j<len(candidates): answer=min(answer,abs(total-2*(a+candidates[j])))
    return answer
```

关键在 `right[half-count]`：左边挑 count 个，右边必须挑 half−count 个，两半人数之和才是 half，另一 half 自动归另一组。这一行就是在"精确排除数量不合法的方案"。每个 a 的目标是让 a+b 贴近 total/2；`(total-2*a+1)//2` 用加一再整除的方式在整数域里取到"上取整的一半"，配合检查前驱后继两种位置，无论最优 b 在目标左侧还是右侧都抓得到。差一律写成 `abs(total-2*(a+b))`——一组拿 a+b、另一组拿 total−(a+b)，差的绝对值就是这个二倍式，全程无浮点。

## 5. 分段实现：按顺序运行

**本章接口：** `subset_sums(nums)`、`min_abs_difference(nums, goal)`、`minimum_difference_equal_size(nums)`

### 导入本章使用的库

In [1]:
from bisect import bisect_left

### subset_sums

In [2]:
def subset_sums(nums):
    sums = [0]
    for x in nums:
        sums += [s + x for s in sums]
    return sums

### min_abs_difference

In [3]:
def min_abs_difference(nums, goal):
    mid = len(nums) // 2
    left = subset_sums(nums[:mid])
    right = sorted(subset_sums(nums[mid:]))
    answer = abs(goal)
    for a in left:
        p = bisect_left(right, goal - a)
        for j in (p - 1, p):
            if 0 <= j < len(right):
                answer = min(answer, abs(a + right[j] - goal))
        if answer == 0:
            return 0
    return answer

### _sums_by_size

In [4]:
def _sums_by_size(nums):
    buckets = [[] for _ in range(len(nums) + 1)]
    buckets[0] = [0]
    for i, x in enumerate(nums):
        for count in range(i + 1, 0, -1):
            buckets[count].extend((s + x for s in buckets[count - 1]))
    return buckets

### minimum_difference_equal_size

In [5]:
def minimum_difference_equal_size(nums):
    if len(nums) % 2:
        raise ValueError('even input length required')
    half = len(nums) // 2
    left = _sums_by_size(nums[:half])
    right = _sums_by_size(nums[half:])
    total = sum(nums)
    answer = float('inf')
    for count, values in enumerate(left):
        candidates = sorted(right[half - count])
        for a in values:
            target = (total - 2 * a + 1) // 2
            p = bisect_left(candidates, target)
            for j in (p - 1, p):
                if 0 <= j < len(candidates):
                    answer = min(answer, abs(total - 2 * (a + candidates[j])))
    return answer

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [6]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[3,9,7,3]; half=len(a)//2; left=_sums_by_size(a[:half]); right=_sums_by_size(a[half:])
show_table(['左侧选取数量','左和候选','对应右侧数量','右和候选'],[(k,left[k],half-k,right[half-k]) for k in range(half+1)])
print('等大小最小差',minimum_difference_equal_size(a))

左侧选取数量,左和候选,对应右侧数量,右和候选
0,[0],2,[10]
1,"[3, 9]",1,"[7, 3]"
2,[12],0,[0]


等大小最小差 2


## 7. 正确性、适用条件与复杂度

每个完整子集唯一分解为左右两个子集；二分目标的前驱或后继必有距离目标最近的右和。等大小问题只组合count与half−count桶，精确排除数量不合法的方案。比较目标使用整数二倍表达式，不把巨大整数转换为浮点。

**代价：** 折半后时间通常O(n·2^(n/2))（含排序/二分），空间O(2^(n/2))；仍是指数算法。subset_sums保留不同子集导致的重复和，不能把它误当作不同和集合。

### 展开理解

**为什么两半配对不漏方案？** 任何一个完整子集都能唯一地切成"它落在左半的部分"和"它落在右半的部分"（每个元素归哪半在切数组时就定死了）。所以枚举左半全部子集 × 在右半找最优搭档，覆盖的正是全部 2^n 个完整子集，一个不漏也没有重复计数的问题。

**为什么二分只看前驱后继就够了？** 右半排序后，对固定目标 t，所有比"前驱"更小的值离 t 更远，所有比"后继"更大的值也离 t 更远——这是有序数列的单调性保证的。所以最小距离必然在这两个候选里。

**为什么等大小版分桶是对的？** 合法方案必须"左 count + 右 half−count"；count 从 0 取到 half 时，右桶下标 half−count 恰好从 half 取到 0，所有合法的数量组合被一一走遍，而任何人数不匹配的组合根本没被生成过，也就无需事后剔除。

**复杂度，代入数字感受一下：** 设 n=40、m=2^20≈10^6。左半枚举 10^6 个和，右半 10^6 个和排序约 10^6×20=2×10^7 次操作；之后每个左和做一次 O(log m)≈20 步的二分，共约 2×10^7 次。总共几千万次操作，一秒内跑完。而完整枚举 2^40≈10^12 次就算每次只要 1 纳秒也要 15 分钟往上。空间上要存右半那 10^6 个和，O(2^(n/2))，几十 MB 内搞定。注意它**仍然是指数级**，n=60 时 2^30≈10^9 就又顶不住了——折半救的是 n≈40 这一档，不是万能钥匙。

**整个流水线分四段，各自的账要分开记：** （1）两侧枚举各 2^(n/2) 次"复制旧和加新数"；（2）排序右半 m log m；（3）逐左和二分 log m；（4）等大小版多了"分桶"这一步——桶内还要各自再排序（代码里 `sorted(right[half-count])` 是循环内做的，同一桶只被排一次，但换桶就重排）。“正确性与复杂度”部分 把总账写成 O(n·2^(n/2))，意思是"指数项 2^(n/2) 乘上一个多项式因子 n"，其中 n 来自排序和二分里都绕不开的 log 因子乘上枚举量级。面试或比赛报复杂度时，说清"哪一段贡献了 2^(n/2)、哪一段贡献了 log"会比只背一个 O 记号更有说服力。

## 8. 单元测试：每个用例在检查什么

- `assert min_abs_difference([5,-7,3,5],6)==0`：存在精确命中的子集（和恰好 6），测正常值 + 提前返回分支。
- `assert minimum_difference_equal_size([3,9,7,3])==2`：“运行示例”部分 的例子，测正常值。
- `assert minimum_difference_equal_size([-36,36])==72`：n=2 的最小规模，一组 {−36}、一组 {36}，差 72。测极小边界。
- `assert minimum_difference_equal_size([])==0`：空数组的退化情形，两半都是空集，差 0。测空输入边界。
- 随机对拍段：n 从 0 到 10，每个 n 随机 20 组数据（元素 −20..20，goal −30..30），用位掩码 `for mask in range(1<<n)` 暴力枚举全部子集和当标准答案，分别核对 `subset_sums`（排序后逐项相等）、`min_abs_difference`（等于暴力最小距离）、偶数 n 时的 `minimum_difference_equal_size`（与 `itertools.combinations` 枚举恰好 n/2 个的下标组合的暴力值相等）。小 n 全子集枚举可行，所以这是穷尽真值。
- 对拍的三条比对方式各不相同，值得留意：`sorted(subset_sums(a))==sorted(sums)` 比的是**多重集合**（和相同但顺序乱、含重复都算过）；`min_abs_difference` 比的是**单个数字**（最小距离）；`combinations` 暴力比的是"从 n 个里恰好挑 n/2 个"的**组合枚举**——它天然保证等大小，正好卡住分桶逻辑的命门。n=10 时全子集 1024 个、组合 252 个，暴力毫无压力；也正因为随机数据带负数、带重复值，重复和与负目标的分支都被踩到了。
- `assert minimum_difference_equal_size([10**30,10**30+1])==1`：两个天文数字，一组差 1。这条专测"整数二倍表达式"——如果代码里用了 `/2` 这种浮点除法，10^30 级别会丢精度，这条就会挂。这是浮点陷阱测试。

In [7]:
assert min_abs_difference([5, -7, 3, 5], 6) == 0

assert minimum_difference_equal_size([3, 9, 7, 3]) == 2

assert minimum_difference_equal_size([-36, 36]) == 72

assert minimum_difference_equal_size([]) == 0

from random import Random

from itertools import combinations

rng = Random(142)

for n in range(11):
    for _ in range(20):
        a = [rng.randrange(-20, 21) for _ in range(n)]
        goal = rng.randrange(-30, 31)
        sums = [sum((a[i] for i in range(n) if mask >> i & 1)) for mask in range(1 << n)]
        assert sorted(subset_sums(a)) == sorted(sums)
        assert min_abs_difference(a, goal) == min((abs(x - goal) for x in sums))
        if n % 2 == 0:
            want = min((abs(sum(a) - 2 * sum((a[i] for i in ids))) for ids in combinations(range(n), n // 2)))
            assert minimum_difference_equal_size(a) == want

assert minimum_difference_equal_size([10 ** 30, 10 ** 30 + 1]) == 1

print('N142: 所有本章断言通过')

N142: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 等大小划分时按选取数量分桶。

**练习 2：** 比较2^n和2^(n/2)的实际状态数。

<details>
<summary>展开思路提示（不是完整答案）</summary>

**练习 1（等大小时按个数分桶）：** 提示：先构造一个"只按和排序配对会得出错误答案"的反例（比如左半有 1 个超大数、右半有 3 个小数的场景，两半各 4 个数时 half=4，你和不在 4 号桶的右和配对就是非法方案）。然后想清楚：桶的下标范围是多少？count 和 half−count 谁配谁？边界是 count=0 和 count=half 两头。最后拿 `itertools.combinations` 的暴力解对拍。

**练习 2（比较 2^n 与 2^(n/2) 的实际状态数）：** 提示：写个小脚本打印 n = 10, 20, 30, 40, 50 时的 2^n、2^(n/2) 和两者比值，再实际跑 `subset_sums` 感受内存和耗时在哪一档开始不可接受。想清楚边界：多大的 n 你的机器能存下 2^(n/2) 个整数（每个 Python int 还不止 8 字节）？这份"实测表"就是你以后判断"该不该上折半"的依据。验收时除了贴表，还要回答一个追问：如果题目限定元素全部非负、且 goal 很小，有没有比折半更省的多项式做法（提示：想想背包型的 DP，值域受限时状态数是 n×goal 而不是 2^n）？这能帮你建立"先看值域、再选算法"的习惯。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [8]:
# 完整实现：本单元格不依赖其他单元格。
from bisect import bisect_left

def subset_sums(nums):
    sums = [0]
    for x in nums:
        sums += [s + x for s in sums]
    return sums

def min_abs_difference(nums, goal):
    mid = len(nums) // 2
    left = subset_sums(nums[:mid])
    right = sorted(subset_sums(nums[mid:]))
    answer = abs(goal)
    for a in left:
        p = bisect_left(right, goal - a)
        for j in (p - 1, p):
            if 0 <= j < len(right):
                answer = min(answer, abs(a + right[j] - goal))
        if answer == 0:
            return 0
    return answer

def _sums_by_size(nums):
    buckets = [[] for _ in range(len(nums) + 1)]
    buckets[0] = [0]
    for i, x in enumerate(nums):
        for count in range(i + 1, 0, -1):
            buckets[count].extend((s + x for s in buckets[count - 1]))
    return buckets

def minimum_difference_equal_size(nums):
    if len(nums) % 2:
        raise ValueError('even input length required')
    half = len(nums) // 2
    left = _sums_by_size(nums[:half])
    right = _sums_by_size(nums[half:])
    total = sum(nums)
    answer = float('inf')
    for count, values in enumerate(left):
        candidates = sorted(right[half - count])
        for a in values:
            target = (total - 2 * a + 1) // 2
            p = bisect_left(candidates, target)
            for j in (p - 1, p):
                if 0 <= j < len(candidates):
                    answer = min(answer, abs(total - 2 * (a + candidates[j])))
    return answer

# 示例与回归测试
assert min_abs_difference([5, -7, 3, 5], 6) == 0

assert minimum_difference_equal_size([3, 9, 7, 3]) == 2

assert minimum_difference_equal_size([-36, 36]) == 72

assert minimum_difference_equal_size([]) == 0

from random import Random

from itertools import combinations

rng = Random(142)

for n in range(11):
    for _ in range(20):
        a = [rng.randrange(-20, 21) for _ in range(n)]
        goal = rng.randrange(-30, 31)
        sums = [sum((a[i] for i in range(n) if mask >> i & 1)) for mask in range(1 << n)]
        assert sorted(subset_sums(a)) == sorted(sums)
        assert min_abs_difference(a, goal) == min((abs(x - goal) for x in sums))
        if n % 2 == 0:
            want = min((abs(sum(a) - 2 * sum((a[i] for i in ids))) for ids in combinations(range(n), n // 2)))
            assert minimum_difference_equal_size(a) == want

assert minimum_difference_equal_size([10 ** 30, 10 ** 30 + 1]) == 1

print('N142: 所有本章断言通过')

N142: 所有本章断言通过


## 11. 代表题与迁移

- **1755. Closest Subsequence Sum（最接近目标值的子序列和）**：就是 `min_abs_difference` 的原题（n≤40，goal 范围大），练的是切半 + 排序 + 二分夹逼这套基本功。
- **2035. Partition Array Into Two Arrays to Minimize Sum Difference（将数组分成两个数组并最小化和的差）**：就是 `minimum_difference_equal_size` 的原题，练的是"按个数分桶 + 整数二倍表达式"这两处相比 1755 多出来的坑。
- **805. Split Array With Same Average（数组的均值分割）**：折半法的延伸应用——把"是否存在子集均值等于整体均值"转化成子集和方程后，同样可以折半枚举两半来解，练的是"把新问题变形成本章模型"的迁移能力。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。